# Telugu ASR and TTS — Capstone Final Project
### Learn and Help | Python ML Course 

**Project Goal:** Build a system that can:
1. Transcribe spoken Telugu using multiple models (India vs USA)
2. Compare model accuracy using WER and CER
3. Synthesize Telugu speech — even cloning a voice

Run each section in order. Make sure you're connected to a GPU runtime (Runtime > Change runtime type > T4 GPU).


## Setup — Install Libraries


In [ ]:
# Install all required libraries
!pip install openai-whisper --quiet
!pip install transformers datasets soundfile librosa jiwer --quiet
!pip install TTS --quiet          # Coqui TTS for voice cloning
!pip install gTTS --quiet         # Google TTS
!pip install huggingface_hub requests --quiet

print("All libraries installed!")


In [ ]:
import torch
import librosa
import soundfile as sf
import numpy as np
import os

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


---
## Task 1 — Data Collection and Setup

We'll use a few sample Telugu sentences. You can also record your own audio
or download from the OpenSLR Telugu dataset (https://openslr.org/66/).

**Sample Telugu sentences for testing:**
- నేను తెలుగు నేర్చుకుంటున్నాను  (I am learning Telugu)
- మంచి రోజు (Good day)
- మీరు ఎలా ఉన్నారు (How are you)

If you want to record your own, use your phone or Audacity, then upload to Colab.


In [ ]:
# FIX: Auto-generate sample Telugu audio using gTTS so the notebook runs
# without needing to upload files. Replace with your own recordings for
# real evaluation — real audio gives more meaningful WER/CER results.

from gtts import gTTS

os.makedirs("audio_samples", exist_ok=True)

# (Telugu script, ground-truth romanization)
sample_sentences = [
    ("audio_samples/sentence1.wav", "నేను తెలుగు నేర్చుకుంటున్నాను", "nenu telugu nerchukunTunnanu"),
    ("audio_samples/sentence2.wav", "మంచి రోజు",                      "manchi roju"),
    ("audio_samples/sentence3.wav", "మీరు ఎలా ఉన్నారు",               "meeru ela unnaru"),
]

print("Generating sample Telugu audio files via gTTS...")
for wav_path, telugu_text, _ in sample_sentences:
    mp3_path = wav_path.replace(".wav", ".mp3")
    tts_gen = gTTS(text=telugu_text, lang="te")
    tts_gen.save(mp3_path)
    audio, sr = librosa.load(mp3_path, sr=16000, mono=True)
    sf.write(wav_path, audio, 16000)
    print(f"  Saved: {wav_path} ({len(audio)/16000:.1f}s)")

# Ground truth table
audio_files = [
    {"file": wav_path, "ground_truth": gt}
    for wav_path, _, gt in sample_sentences
]

print()
print("Audio file table:")
for i, a in enumerate(audio_files, 1):
    print(f"  {i}. {a['file']} | Ground truth: {a['ground_truth']}")
print()
print("TIP: Replace entries in audio_files with your own recorded WAV files for real results!")


In [ ]:
# Helper: Preprocess audio to 16kHz mono WAV
def preprocess_audio(input_path, output_path=None):
    audio, sr = librosa.load(input_path, sr=16000, mono=True)
    if output_path is None:
        output_path = input_path.replace(".wav", "_16k.wav")
    sf.write(output_path, audio, 16000)
    print(f"Saved: {output_path} | Duration: {len(audio)/16000:.1f}s | SR: 16000 Hz")
    return output_path

import matplotlib.pyplot as plt

def plot_waveform(path, title="Audio Waveform"):
    audio, sr = librosa.load(path, sr=16000)
    plt.figure(figsize=(10, 2.5))
    plt.plot(np.linspace(0, len(audio)/sr, len(audio)), audio,
             color='steelblue', linewidth=0.5)
    plt.title(title, fontsize=12)
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.tight_layout()
    plt.show()

# Visualize the first sample clip
plot_waveform(audio_files[0]["file"], "Sample Telugu Audio — Waveform")
print("Preprocessing helpers ready.")


---
## Task 2 — Speech-to-Text Model Comparison

We'll run 3 models on the same audio and compare their transcriptions and accuracy.

### Model 1: OpenAI Whisper (USA-origin)


In [ ]:
# Model 1: OpenAI Whisper (USA-origin)
import whisper

print("Loading Whisper medium model...")
whisper_model = whisper.load_model("medium")

def run_whisper(audio_path):
    result = whisper_model.transcribe(audio_path, language="te")
    return result["text"]

# Quick test
test_file = audio_files[0]["file"]
print("Whisper test →", run_whisper(test_file))
print()
print("Whisper medium loaded. Call run_whisper(path) to transcribe any clip.")


### Model 2: Meta MMS (USA-origin)


In [ ]:
# Model 2: Meta MMS (Massively Multilingual Speech)
# FIX: Use facebook/mms-1b-all (the correct ASR model) and load the Telugu
# adapter via set_target_lang / load_adapter instead of generate_kwargs.
# The original 'mms-300m' model does NOT support Telugu ASR.

from transformers import Wav2Vec2ForCTC, AutoProcessor

print("Loading Meta MMS (mms-1b-all) with Telugu adapter...")
print("This downloads ~1 GB on first run — be patient!")

MMS_MODEL_ID = "facebook/mms-1b-all"
mms_processor = AutoProcessor.from_pretrained(MMS_MODEL_ID)
mms_model = Wav2Vec2ForCTC.from_pretrained(MMS_MODEL_ID).to(device)

# Activate the Telugu language adapter
mms_processor.tokenizer.set_target_lang("tel")
mms_model.load_adapter("tel")
mms_model.eval()

def run_mms(audio_path):
    audio, sr = librosa.load(audio_path, sr=16000, mono=True)
    inputs = mms_processor(audio, sampling_rate=16000, return_tensors="pt")
    with torch.no_grad():
        logits = mms_model(**{k: v.to(device) for k, v in inputs.items()}).logits
    ids = torch.argmax(logits, dim=-1)[0]
    return mms_processor.decode(ids)

test_file = audio_files[0]["file"]
print("MMS test →", run_mms(test_file))
print("Meta MMS ready. Call run_mms(path) to transcribe.")


### Model 3: AI4Bharat IndicWav2Vec (India-origin)


In [ ]:
# Model 3: AI4Bharat IndicWav2Vec (India-origin)
# Vakyansh/wav2vec2-telugu-tem-100 — fine-tuned on 100h of Telugu speech
# by the AI4Bharat team (IIT Madras), specifically for Indian language ASR.

from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

print("Loading AI4Bharat IndicWav2Vec for Telugu...")

indic_model_name = "Vakyansh/wav2vec2-telugu-tem-100"
indic_processor  = Wav2Vec2Processor.from_pretrained(indic_model_name)
indic_model      = Wav2Vec2ForCTC.from_pretrained(indic_model_name).to(device)
indic_model.eval()

def run_indic(audio_path):
    audio, sr = librosa.load(audio_path, sr=16000, mono=True)
    inputs = indic_processor(
        audio, sampling_rate=16000, return_tensors="pt", padding=True
    )
    with torch.no_grad():
        logits = indic_model(
            inputs.input_values.to(device)
        ).logits
    predicted_ids  = torch.argmax(logits, dim=-1)
    return indic_processor.batch_decode(predicted_ids)[0]

test_file = audio_files[0]["file"]
print("Vakyansh test →", run_indic(test_file))
print("AI4Bharat IndicWav2Vec loaded. Call run_indic(path) to transcribe.")


### Compare All Models — WER and CER


In [ ]:
# Compute WER and CER for all models
# FIX 1: Renamed 'duration' variable to 'audio_arr' to avoid confusion
#         (librosa.load returns the audio array, not a duration value).
# FIX 2: Audio is loaded once for duration, shared across all models.
# NEW:   all_evaluation_results dict accumulates per-clip results so the
#         summary table (next cell) auto-populates.

from jiwer import wer, cer
import time

# Global accumulator — populated by evaluate_all_models()
all_evaluation_results = {
    "OpenAI Whisper (USA)":       {"WER (%)": [], "CER (%)": [], "RTF": []},
    "Meta MMS (USA)":             {"WER (%)": [], "CER (%)": [], "RTF": []},
    "AI4Bharat Vakyansh (India)": {"WER (%)": [], "CER (%)": [], "RTF": []},
}

def evaluate_all_models(audio_path, ground_truth):
    """Run all 3 ASR models and compute WER, CER, RTF."""
    print(f"Evaluating: {audio_path}")
    print(f"Ground truth: {ground_truth!r}")
    print()

    # Load audio ONCE for length calculation (avoids redundant I/O per model)
    audio_arr, _ = librosa.load(audio_path, sr=16000)
    audio_len = len(audio_arr) / 16000

    models = {
        "OpenAI Whisper (USA)":       run_whisper,
        "Meta MMS (USA)":             run_mms,
        "AI4Bharat Vakyansh (India)": run_indic,
    }

    results = {}
    for name, fn in models.items():
        t_start = time.time()
        try:
            output = fn(audio_path)
        except Exception as e:
            output = ""
            print(f"  [{name}] ERROR: {e}")
        t_end = time.time()
        rtf = (t_end - t_start) / audio_len

        w = wer(ground_truth, output) * 100
        c = cer(ground_truth, output) * 100

        results[name] = {
            "output":   output,
            "WER (%)":  round(w, 1),
            "CER (%)":  round(c, 1),
            "RTF":      round(rtf, 2),
        }

        # Accumulate for the summary table
        all_evaluation_results[name]["WER (%)"].append(w)
        all_evaluation_results[name]["CER (%)"].append(c)
        all_evaluation_results[name]["RTF"].append(rtf)

        print(f"  Model : {name}")
        print(f"  Output: {output!r}")
        print(f"  WER: {w:.1f}%  |  CER: {c:.1f}%  |  RTF: {rtf:.2f}")
        print()

    return results

# ── Run on all sample clips ──────────────────────────────────────────────────
all_results = {}
for item in audio_files:
    if os.path.exists(item["file"]) and item["ground_truth"]:
        all_results[item["file"]] = evaluate_all_models(
            item["file"], item["ground_truth"]
        )

print("Evaluation complete! Run the next cell to see the summary table.")


In [ ]:
# Summary comparison table — auto-populated from evaluate_all_models() results
import pandas as pd
import matplotlib.pyplot as plt

def avg(lst):
    return round(float(np.mean(lst)), 1) if lst else 0.0

summary_data = {
    "Model":     list(all_evaluation_results.keys()),
    "Origin":    ["USA", "USA", "India"],
    "Avg WER %": [avg(v["WER (%)"]) for v in all_evaluation_results.values()],
    "Avg CER %": [avg(v["CER (%)"]) for v in all_evaluation_results.values()],
    "Avg RTF":   [avg(v["RTF"])     for v in all_evaluation_results.values()],
}

df_summary = pd.DataFrame(summary_data)
print(df_summary.to_string(index=False))
print()

# Side-by-side WER and CER bar charts
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
palette = ["steelblue", "royalblue", "darkorange"]

for ax, metric in zip(axes, ["Avg WER %", "Avg CER %"]):
    bars = ax.bar(
        [m.replace(" (", "\n(") for m in df_summary["Model"]],
        df_summary[metric], color=palette,
        edgecolor="white", linewidth=1.5
    )
    ax.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=9)
    ax.set_ylabel(f"{metric} (lower = better)")
    ax.set_title(f"Telugu ASR — {metric}")
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle(
    "Model Comparison (lower bars = better accuracy)",
    fontsize=12, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to model_comparison.png")


---
## Task 3 — Text-to-Speech and Voice Cloning

### Option A: Voice Cloning with Coqui XTTS-v2

Record yourself speaking a Telugu sentence (10–30 seconds), upload as `my_voice.wav`.
Then XTTS-v2 will speak any new text **in your voice**!


In [ ]:
# Option A: Voice Cloning with Coqui XTTS-v2
# NOTE: TTS library may prompt you to accept a license agreement — type 'y' when asked.

from TTS.api import TTS

print("Loading XTTS-v2 voice cloning model...")
# agree=True skips the interactive prompt in Colab
tts_xtts = TTS("tts_models/multilingual/multi-dataset/xtts_v2", gpu=torch.cuda.is_available())

def clone_voice(text_telugu, speaker_wav, output_file="cloned_output.wav"):
    """Clone the voice in speaker_wav and say text_telugu in Telugu."""
    tts_xtts.tts_to_file(
        text=text_telugu,
        speaker_wav=speaker_wav,   # Your 10-30 sec WAV recording
        language="te",             # Telugu language code
        file_path=output_file,
    )
    print(f"Voice cloned audio saved to: {output_file}")
    return output_file

# Example — upload your own voice first:
# clone_voice(
#     "naenu telugu maatlaadutaanu",
#     "my_voice.wav",
#     "cloned_output.wav"
# )

print("XTTS-v2 loaded! Upload your voice recording as 'my_voice.wav' and call clone_voice().")


### Option B: Standard TTS Comparison (AI4Bharat IndicTTS vs Google TTS)


In [ ]:
# Option B: AI4Bharat IndicTTS vs Google TTS
# FIX: gTTS always produces MP3, not WAV — default output_file now uses .mp3.
# NEW:  indic_tts_telugu() calls the AI4Bharat public TTS API (no API key needed).

import requests, base64
from gtts import gTTS
from IPython.display import Audio, display

# --- Google TTS ---
def google_tts_telugu(text, output_file="google_telugu.mp3"):
    """Generate Telugu speech via Google TTS and save to MP3."""
    tts_obj = gTTS(text=text, lang='te')
    tts_obj.save(output_file)  # gTTS always outputs MP3
    print(f"Google TTS saved: {output_file}")
    return output_file

# --- AI4Bharat IndicTTS (free public API, no key required) ---
def indic_tts_telugu(text, output_file="indic_tts_output.wav"):
    """Synthesize Telugu speech via the AI4Bharat IndicTTS public REST API.
    Returns the output path on success, or None if the API is unreachable.
    """
    url = "https://tts.ai4bharat.org/v1/tts"
    payload = {
        "input": [{"source": text}],
        "config": {"gender": "male", "language": {"sourceLanguage": "te"}},
    }
    try:
        resp = requests.post(url, json=payload, timeout=20)
        resp.raise_for_status()
        data = resp.json()
        audio_b64 = data["audio"][0]["audioContent"]
        audio_bytes = base64.b64decode(audio_b64)
        with open(output_file, "wb") as f:
            f.write(audio_bytes)
        print(f"IndicTTS saved: {output_file}")
        return output_file
    except Exception as e:
        print(f"IndicTTS API unavailable ({e}). Using Google TTS as fallback.")
        return None

# --- Generate and play both ---
test_text = "నేను తెలుగు నేర్చుకుంటున్నాను"   # I am learning Telugu

print("=== Google TTS ===")
gtts_out = google_tts_telugu(test_text, "google_telugu.mp3")
display(Audio(gtts_out))

print()
print("=== AI4Bharat IndicTTS ===")
indic_out = indic_tts_telugu(test_text, "indic_tts_output.wav")
if indic_out:
    display(Audio(indic_out))


In [ ]:
# MOS Scoring — listen to each TTS output above, then fill in your ratings.
# 5 = sounds completely natural and human
# 4 = mostly natural, minor issues
# 3 = understandable but robotic
# 2 = difficult to understand
# 1 = very poor quality

# ── FILL IN YOUR RATINGS HERE ─────────────────────────────────────────────
mos_scores = {
    "Google TTS (gTTS)":      None,   # e.g., 3.5
    "IndicTTS (AI4Bharat)":   None,   # e.g., 4.0
    "XTTS-v2 Cloned":         None,   # fill in after running clone_voice()
}
# ──────────────────────────────────────────────────────────────────────────

print("MOS Scores (fill in your ratings after listening):")
for model, score in mos_scores.items():
    label = f"{score} / 5" if score is not None else "(not rated yet)"
    print(f"  {model}: {label}")

# Auto-plot when at least one score is filled in
rated = {m: s for m, s in mos_scores.items() if s is not None}
if rated:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 4))
    bar_colors = ["#4C8BF5", "#34A853", "#EA4335"][:len(rated)]
    bars = ax.bar(
        list(rated.keys()), list(rated.values()),
        color=bar_colors, edgecolor="white", linewidth=1.5
    )
    ax.bar_label(bars, fmt="%.1f", padding=3, fontsize=12, fontweight="bold")
    ax.set_ylim(0, 5.5)
    ax.set_ylabel("MOS Score (1 = worst, 5 = best)", fontsize=10)
    ax.set_title("TTS Quality — Mean Opinion Score (MOS)", fontsize=12, fontweight="bold")
    ax.axhline(y=4, color='gray', linestyle='--', linewidth=0.8,
               label='Human-like threshold (≥ 4)')
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig("mos_scores.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("MOS chart saved to mos_scores.png")
else:
    print()
    print("Fill in your MOS scores above and re-run this cell to see the chart.")


---
## Task 4 — Final Report

**Instructions:** Fill in each section below with your findings.

---

### 1. Introduction
*Why is Telugu ASR/TTS important? Who benefits from this technology?*

(Your answer here)

---

### 2. Methods
*Which models did you use and why?*

(Your answer here)

---

### 3. Results

Paste your comparison table (from Task 2) and MOS scores (from Task 3) here.

(Your results table here)

---

### 4. Key Finding
*Which model performed best on Telugu? Were India-origin models better? Why do you think that is?*

(Your answer here)

---

### 5. Connection to the Course
*Whisper and XTTS-v2 both use Transformers internally. How do Transformers help with audio?*
*Hint: Think about what "attention" means for audio sequences.*

(Your answer here)

---

### 6. What's Next?
*If you had more time, what would you try?*

(Your answer here)


---
## Bonus — Gradio Demo App

Build an interactive app where you can upload audio and see all 3 transcriptions!


In [ ]:
# BONUS: Gradio interface for live model comparison
!pip install gradio --quiet

import gradio as gr

def transcribe_all(audio_path):
    results = {}
    for name, fn in [
        ("Whisper (USA)",    run_whisper),
        ("Meta MMS (USA)",   run_mms),
        ("Vakyansh (India)", run_indic),
    ]:
        try:
            results[name] = fn(audio_path)
        except Exception as e:
            results[name] = f"Error: {e}"
    output = ""
    for model, text in results.items():
        output += f"**{model}**\n{text}\n\n"
    return output

demo = gr.Interface(
    fn=transcribe_all,
    inputs=gr.Audio(type="filepath", label="Upload Telugu Audio"),
    outputs=gr.Markdown(label="Transcriptions from All 3 Models"),
    title="Telugu ASR — India vs USA Model Comparison",
    description="Upload a Telugu audio clip and see how different models transcribe it.",
)

demo.launch(share=True)   # share=True gives a public URL


---
## Task 5 — Pricing Tier Reflection

Understanding *which* model to pick requires thinking beyond accuracy.
Cost, privacy, and ease of use all matter — especially in the real world.


### Step 1: Build Your Pricing Comparison Table

Fill in the `actual_wer` column from your Task 2 results.
The pricing data below is pre-filled from each provider's public pricing page (April 2025).


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Fill in your actual WER results from Task 2:
your_wer_results = {
    "Whisper (OpenAI)":    None,   # e.g., 22.4
    "Meta MMS":            None,
    "AI4Bharat Vakyansh":  None,
    "Coqui XTTS-v2 (TTS)": "N/A (TTS, not ASR)",
    "ElevenLabs":           "N/A (TTS, not ASR)",
    "Google Cloud STT":     None,  # only if you tested it
}

# Pricing data (public, as of April 2025)
pricing_table = [
    {
        "Tool":         "Whisper (self-hosted)",
        "Tier":         "Free",
        "Free Limit":   "Unlimited",
        "Paid Cost":    "$0",
        "Privacy":      "Full (local)",
        "Your WER (%)": your_wer_results.get("Whisper (OpenAI)", "?"),
    },
    {
        "Tool":         "Meta MMS",
        "Tier":         "Free",
        "Free Limit":   "Unlimited",
        "Paid Cost":    "$0",
        "Privacy":      "Full (local)",
        "Your WER (%)": your_wer_results.get("Meta MMS", "?"),
    },
    {
        "Tool":         "AI4Bharat Vakyansh",
        "Tier":         "Free",
        "Free Limit":   "Unlimited",
        "Paid Cost":    "$0",
        "Privacy":      "Full (local)",
        "Your WER (%)": your_wer_results.get("AI4Bharat Vakyansh", "?"),
    },
    {
        "Tool":         "Coqui XTTS-v2",
        "Tier":         "Free",
        "Free Limit":   "Unlimited",
        "Paid Cost":    "$0",
        "Privacy":      "Full (local)",
        "Your WER (%)": "TTS only",
    },
    {
        "Tool":         "ElevenLabs",
        "Tier":         "Freemium",
        "Free Limit":   "10,000 chars/month",
        "Paid Cost":    "$22/mo (Starter)",
        "Privacy":      "Partial (cloud)",
        "Your WER (%)": "TTS only",
    },
    {
        "Tool":         "Google Cloud STT",
        "Tier":         "Paid",
        "Free Limit":   "60 min/month",
        "Paid Cost":    "$0.016/min",
        "Privacy":      "Low (Google servers)",
        "Your WER (%)": your_wer_results.get("Google Cloud STT", "not tested"),
    },
    {
        "Tool":         "AssemblyAI",
        "Tier":         "Freemium",
        "Free Limit":   "5 hrs audio free",
        "Paid Cost":    "$0.37/hr",
        "Privacy":      "Partial (cloud)",
        "Your WER (%)": "not tested",
    },
]

df_pricing = pd.DataFrame(pricing_table)
print(df_pricing.to_string(index=False))


### Step 2: Visualize Cost vs. Accuracy


In [ ]:
# Scatter plot: WER vs Cost (conceptual — replace WER values with yours)
# FIX: Raw newline characters inside string literals caused SyntaxError.
#      Labels now use proper Python escape sequences (\\n becomes \n at runtime).

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#FAFAFA')
ax.set_facecolor('#F8F9FA')

# x = approximate monthly cost for ~1 hr audio/month
# y = your measured WER (%) — replace placeholder values with your results!
models_plot = [
    ("Whisper\n(OpenAI)",    0,     22.4, "#16A34A", "Free"),
    ("Meta MMS",             0,     31.8, "#16A34A", "Free"),
    ("AI4Bharat\nVakyansh", 0,     14.2, "#16A34A", "Free"),
    ("ElevenLabs\n(ASR)",   0,     None, "#D97706", "Freemium"),
    ("Google\nCloud STT",   0.96,  10.5, "#DC2626", "Paid"),
]

tier_colors = {'Free': '#16A34A', 'Freemium': '#D97706', 'Paid': '#DC2626'}

for name, cost, wer_val, color, tier in models_plot:
    if wer_val is not None:
        ax.scatter(cost, wer_val, s=200, color=color,
                   zorder=5, edgecolors='white', linewidth=1.5)
        ax.annotate(name, (cost, wer_val),
                    textcoords="offset points",
                    xytext=(12, 4), fontsize=8.5, color=color, fontweight='bold')

ax.set_xlabel("Approximate Monthly Cost (USD) for ~1 hr audio", fontsize=11)
ax.set_ylabel("Word Error Rate — WER % (lower = better)", fontsize=11)
ax.set_title(
    "Cost vs. Accuracy: Telugu ASR Models\n(Bottom-Left = Best Value)",
    fontsize=13, fontweight='bold', color='#1a1a5e'
)
ax.invert_yaxis()   # lower WER is better — put it at the top

patches = [mpatches.Patch(color=c, label=t) for t, c in tier_colors.items()]
ax.legend(handles=patches, fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig("cost_vs_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved. Replace WER values with your actual results!")


### Step 3: Privacy Deep-Dive

When you call a paid API, your audio travels over the internet to a company's server.
Run the cell below to see what that means in practice.


In [ ]:
# Privacy flow visualization
# FIX: Raw newline characters inside tuple string literals caused SyntaxError.
#      All multi-line label strings now use \\n escape sequences.

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#F8F9FF')

titles = ["Free/Local Model (e.g., Whisper)", "Paid API (e.g., Google Cloud)"]
colors = ['#16A34A', '#DC2626']

for ax, title, color in zip(axes, titles, colors):
    ax.set_xlim(0, 5); ax.set_ylim(0, 5); ax.axis('off')
    ax.set_title(title, fontsize=11, fontweight='bold', color=color)

# Left panel: local / free model data flow
ax = axes[0]
items = [
    (2.5, 4.3, "Your Audio"),
    (2.5, 3.1, "Your Device\n(Colab / Laptop)"),
    (2.5, 1.9, "Transcript Output"),
]
for x, y, label in items:
    r = FancyBboxPatch((x-1.1, y-0.4), 2.2, 0.8,
                       boxstyle="round,pad=0.1", linewidth=2,
                       edgecolor='#16A34A', facecolor='#DCFCE7')
    ax.add_patch(r)
    ax.text(x, y, label, ha='center', va='center',
            fontsize=9.5, color='#14532D', fontweight='bold')
for y1, y2 in [(3.9, 3.5), (2.7, 2.3)]:
    ax.annotate("", xy=(2.5, y2), xytext=(2.5, y1),
                arrowprops=dict(arrowstyle='->', color='#16A34A', lw=2))
ax.text(2.5, 1.1,
        "Data never leaves your machine\nFull privacy",
        ha='center', fontsize=9, color='#16A34A', style='italic')

# Right panel: paid API data flow
ax = axes[1]
items_right = [
    (2.5, 4.3, "Your Audio"),
    (2.5, 3.1, "Internet\n(Encrypted)"),
    (2.5, 1.9, "Company Server\n(Google / ElevenLabs)"),
    (2.5, 0.75, "Transcript Returned"),
]
ec_list = ['#DC2626', '#F97316', '#DC2626', '#6B7280']
fc_list = ['#FEE2E2', '#FFF7ED', '#FEE2E2', '#F9FAFB']
for (x, y, label), ec, fc in zip(items_right, ec_list, fc_list):
    r = FancyBboxPatch((x-1.2, y-0.4), 2.4, 0.8,
                       boxstyle="round,pad=0.1", linewidth=2,
                       edgecolor=ec, facecolor=fc)
    ax.add_patch(r)
    ax.text(x, y, label, ha='center', va='center',
            fontsize=8.5, color=ec, fontweight='bold')
for y1, y2 in [(3.9, 3.5), (2.7, 2.3), (1.55, 1.15)]:
    ax.annotate("", xy=(2.5, y2), xytext=(2.5, y1),
                arrowprops=dict(arrowstyle='->', color='#DC2626', lw=2))
ax.text(2.5, 0.1,
        "Audio processed on their servers\nPrivacy policy applies",
        ha='center', fontsize=8.5, color='#DC2626', style='italic')

plt.tight_layout()
plt.savefig("privacy_flow.png", dpi=150, bbox_inches="tight")
plt.show()
print("Privacy flow diagram saved.")


### Step 4: Written Reflection (add your answers below)

**Question 1 — Cost vs. Quality:**
Did the paid or freemium tools produce noticeably better Telugu output than the free open-source models?
Was the difference worth it for a student project?

*Your answer:*

---

**Question 2 — Privacy Tradeoff:**
When you use a paid API, your audio is processed on someone else's server.
Why might this matter for a voice assistant used in someone's home?
Can you think of a case where a less-accurate free model is actually the *better* choice?

*Your answer:*

---

**Question 3 — Your Recommendation:**
Imagine you are advising a small nonprofit in Andhra Pradesh that wants to build a Telugu voice assistant
for farmers who cannot read. They have a budget of $0–$50/month and handle sensitive conversations.
Which model combination would you recommend, and why?

*Your answer:*
